# Model Training on caloric_intake_rice_age_dataset_385

This notebook trains 6 models on the dataset:
- Decision Tree
- XGBoost
- Random Forest
- Ensemble Method (Soft Voting)
- SVM
- Logistic Regression

In [ ]:
%pip install -q numpy==1.26.4 pandas==2.2.2 scikit-learn==1.5.1 xgboost==2.1.4

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

In [ ]:
dataset_path = 'caloric_intake_rice_age_dataset_385.csv'
df = pd.read_csv(dataset_path)
df.head()

In [ ]:
X = df.drop(columns=['class'])
y = df['class']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'XGBoost': XGBClassifier(
        objective='multi:softprob',
        num_class=y.nunique(),
        eval_metric='mlogloss',
        n_estimators=150,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42),
    'Ensemble Method': VotingClassifier(
        estimators=[
            ('lr', LogisticRegression(max_iter=2000, random_state=42)),
            ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
            ('dt', DecisionTreeClassifier(random_state=42)),
        ],
        voting='soft'
    ),
    'SVM': SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=3000, random_state=42),
}

In [ ]:
results = []
reports = {}

for name, model in models.items():
    clf = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    results.append({'Model': name, 'Accuracy': acc, 'Weighted F1': f1})
    reports[name] = classification_report(y_test, y_pred, digits=4)

results_df = pd.DataFrame(results).sort_values(by='Weighted F1', ascending=False).reset_index(drop=True)
results_df

In [ ]:
for model_name, report in reports.items():
    print(f'===== {model_name} =====')
    print(report)